In [1]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
import os

warnings.filterwarnings('ignore')

os.makedirs('../plots', exist_ok=True)

# Load dataset
df = pd.read_csv('../data/home_vehicle_loans.csv')

# Create figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

fig.suptitle(
    'Home & Vehicle Loan Default EDA',
    fontsize=14,
    fontweight='bold'
)

# -----------------------------
# Chart 1: Default by loan type
# -----------------------------
type_def = df.groupby('loan_type')['defaulted'].mean() * 100

colors_t = ['#1D4ED8','#EF4444']

bars = axes[0,0].bar(
    type_def.index,
    type_def.values,
    color=colors_t,
    edgecolor='black',
    alpha=0.85
)

axes[0,0].set_title(
    'Default Rate by Loan Type',
    fontweight='bold'
)

axes[0,0].set_ylabel('Default Rate (%)')

for bar, val in zip(bars, type_def.values):
    axes[0,0].text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+0.1,
        f'{val:.1f}%',
        ha='center',
        fontweight='bold'
    )

# -----------------------------
# Chart 2: Default by LTV
# -----------------------------
df['ltv_bucket'] = pd.cut(
    df['ltv'],
    bins=[0,0.6,0.7,0.8,0.9,1.0],
    labels=['<60%','60-70%','70-80%','80-90%','90%+']
)

ltv_def = df.groupby(
    'ltv_bucket',
    observed=True
)['defaulted'].mean() * 100

colors_l = ['#16A34A','#65A30D','#EAB308','#F97316','#EF4444']

bars2 = axes[0,1].bar(
    ltv_def.index,
    ltv_def.values,
    color=colors_l,
    edgecolor='black',
    alpha=0.85
)

axes[0,1].set_title(
    'Default Rate by LTV Bucket',
    fontweight='bold'
)

axes[0,1].set_ylabel('Default Rate (%)')
axes[0,1].set_xlabel('LTV Ratio')

for bar, val in zip(bars2, ltv_def.values):
    axes[0,1].text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+0.1,
        f'{val:.1f}%',
        ha='center',
        fontweight='bold',
        fontsize=9
    )

# -----------------------------
# Chart 3: CIBIL distribution
# -----------------------------
axes[0,2].hist(
    df[df['defaulted']==0]['cibil'],
    bins=40,
    alpha=0.6,
    color='green',
    label='No Default',
    density=True
)

axes[0,2].hist(
    df[df['defaulted']==1]['cibil'],
    bins=40,
    alpha=0.6,
    color='red',
    label='Default',
    density=True
)

axes[0,2].set_title(
    'CIBIL Score: Default vs No Default',
    fontweight='bold'
)

axes[0,2].set_xlabel('CIBIL Score')

axes[0,2].axvline(
    650,
    color='black',
    linestyle='--',
    linewidth=2,
    label='650 threshold'
)

axes[0,2].legend()

# -----------------------------
# Chart 4: EMI/Income
# -----------------------------
axes[1,0].hist(
    df[df['defaulted']==0]['emi_income'].clip(0,0.8),
    bins=40,
    alpha=0.6,
    color='green',
    label='No Default',
    density=True
)

axes[1,0].hist(
    df[df['defaulted']==1]['emi_income'].clip(0,0.8),
    bins=40,
    alpha=0.6,
    color='red',
    label='Default',
    density=True
)

axes[1,0].axvline(
    0.40,
    color='black',
    linestyle='--',
    linewidth=2,
    label='40% threshold'
)

axes[1,0].set_title(
    'EMI/Income Ratio: Default vs No Default',
    fontweight='bold'
)

axes[1,0].set_xlabel('EMI / Monthly Income')

axes[1,0].legend()

# -----------------------------
# Chart 5: Employment
# -----------------------------
emp_def = df.groupby('employment')['defaulted'].mean() * 100

axes[1,1].bar(
    emp_def.index,
    emp_def.values,
    color=['#1D4ED8','#7C3AED','#16A34A'],
    edgecolor='black',
    alpha=0.85
)

axes[1,1].set_title(
    'Default Rate by Employment Type',
    fontweight='bold'
)

axes[1,1].set_ylabel('Default Rate (%)')

for i, (idx, val) in enumerate(emp_def.items()):
    axes[1,1].text(
        i,
        val+0.1,
        f'{val:.1f}%',
        ha='center',
        fontweight='bold'
    )

# -----------------------------
# Chart 6: Time-to-event
# -----------------------------
axes[1,2].hist(
    df[df['defaulted']==1]['time_to_event'],
    bins=40,
    color='red',
    edgecolor='white',
    alpha=0.85,
    label='Defaulters'
)

axes[1,2].hist(
    df[df['defaulted']==0]['time_to_event'].sample(5000),
    bins=40,
    color='green',
    edgecolor='white',
    alpha=0.5,
    label='No Default (sample)'
)

axes[1,2].set_title(
    'Time to Event Distribution (months)',
    fontweight='bold'
)

axes[1,2].set_xlabel('Months')

axes[1,2].legend()

# -----------------------------
# Save plot
# -----------------------------
plt.tight_layout()

plt.savefig(
    '../plots/loan_eda.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/loan_eda.png")

print()
print("YOUR 3 INTERVIEW INSIGHTS:")

high_ltv = df[df['ltv']>0.85]['defaulted'].mean()*100
low_ltv = df[df['ltv']<=0.60]['defaulted'].mean()*100

print(
    f"1. LTV>85% has {high_ltv:.1f}% default rate vs {low_ltv:.1f}% for LTV<60%"
)

bad_cibil = df[df['cibil']<650]['defaulted'].mean()*100
good_cibil = df[df['cibil']>750]['defaulted'].mean()*100

print(
    f"2. CIBIL<650 has {bad_cibil:.1f}% default vs {good_cibil:.1f}% for CIBIL>750"
)

veh_def = df[df['loan_type']=='Vehicle']['defaulted'].mean()*100
home_def = df[df['loan_type']=='Home']['defaulted'].mean()*100

print(
    f"3. Vehicle loans default {veh_def:.1f}% vs Home loans {home_def:.1f}%"
)

Saved: plots/loan_eda.png

YOUR 3 INTERVIEW INSIGHTS:
1. LTV>85% has 32.4% default rate vs 13.9% for LTV<60%
2. CIBIL<650 has 30.6% default vs 9.4% for CIBIL>750
3. Vehicle loans default 21.4% vs Home loans 16.7%
